# LinearGridTransformer — Industrial-Grade Grid Training & Evaluation
**Architecture:** Taylor Feature Map Linear Attention vs. CDLinear Baseline  
**Hardware Target:** Google Colab GPU (Tesla T4 / V100 / A100)  
**Protocol:** Zero-Leakage Evaluation (Model selection on validation only; final testing on untouched holdout)

In [ ]:
# Cell 1: Verify GPU & Mount Google Drive for Persistent Artifact Storage
import torch
from google.colab import drive

print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA available: True | Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not detected! Enable via Runtime -> Change runtime type -> T4 GPU.")

# Mount Google Drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/LinearGridTetouan/checkpoints
!mkdir -p /content/drive/MyDrive/LinearGridTetouan/results
!mkdir -p /content/drive/MyDrive/LinearGridTetouan/mlflow

In [ ]:
# Cell 2: Clone or Update the Repository
%cd /content
import os
if not os.path.isdir("LinearGridTransformer"):
    !git clone https://github.com/sriram615/LinearGridTransformer.git
%cd /content/LinearGridTransformer

# Install dependencies
!pip install -q pandas psutil mlflow psycopg gridstatus requests

In [ ]:
# Cell 3: Configure Paths & Verify Canonical Grid Dataset
from pathlib import Path
import pandas as pd

# Use the regional grid CSV (or copy from Google Drive)
csv_path = Path("data/raw/grid/grid_power_consumption.csv")
if not csv_path.is_file():
    print("Dataset not found locally; running ETL download...")
    !python3 -m src.data.grid_etl --year 2023 --output data/raw/grid/grid_power_consumption.csv

df = pd.read_csv(csv_path)
print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print("Columns:", df.columns.tolist())
assert df.shape[1] == 9, "Expected 9 columns (DateTime + 5 Weather + 3 Zones)"
assert df.isna().sum().sum() == 0, "Found missing values!"

In [ ]:
# Cell 4: Preprocess Dataset with Train-Only Normalization
# Fits z-score normalization strictly on the training partition [0, 70%)
from src.data.tetouan_preprocess import preprocess_tetouan

artifact_dir = Path("data/processed/grid")
meta = preprocess_tetouan(csv_path, output_dir=artifact_dir)
print(f"Processed metadata written to {artifact_dir}")
print("Train partition rows:", meta["split"]["partitions"]["train"]["row_count"])
print("Validation partition rows:", meta["split"]["partitions"]["validation"]["row_count"])
print("Test partition rows:", meta["split"]["partitions"]["test"]["row_count"])

In [ ]:
# Cell 5: Execute Multi-Seed Comparative Training (Seeds 42, 123, 2026)
# Runs CDLinear and LinearGridTransformer under identical seed conditions
import os
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:////content/drive/MyDrive/LinearGridTetouan/mlflow/tracking.db"
os.environ["MLFLOW_ARTIFACT_ROOT"] = "/content/drive/MyDrive/LinearGridTetouan/mlflow/artifacts"

seeds = [42, 123, 2026]
for seed in seeds:
    print(f"
" + "="*60)
    print(f"Launching Training Run for Seed {seed}")
    print("="*60)
    !python3 -m src.training.train_tetouan \n        --epochs 50 \n        --batch-size 32 \n        --learning-rate 0.001 \n        --weight-decay 0.01 \n        --seed {seed} \n        --artifact-dir data/processed/grid \n        --checkpoint-dir /content/drive/MyDrive/LinearGridTetouan/checkpoints \n        --results-dir /content/drive/MyDrive/LinearGridTetouan/results \n        --run-name grid_colab_s{seed}

In [ ]:
# Cell 6: Zero-Leakage Model Family Selection (Validation Loss ONLY)
# Compare validation scores to choose winning architecture before reading test partition
import json
from pathlib import Path

results_dir = Path("/content/drive/MyDrive/LinearGridTetouan/results")
val_scores = {"CDLinear": [], "LinearGridTransformer": []}

for seed in [42, 123, 2026]:
    manifest_path = results_dir / f"grid_colab_s{seed}_manifest.json"
    data = json.loads(manifest_path.read_text())
    for model in ["CDLinear", "LinearGridTransformer"]:
        best_val = data["models"][model]["best_validation_loss"]
        val_scores[model].append(best_val)

cdl_mean_val = sum(val_scores["CDLinear"]) / len(val_scores["CDLinear"])
lgt_mean_val = sum(val_scores["LinearGridTransformer"]) / len(val_scores["LinearGridTransformer"])

print(f"CDLinear 3-Seed Mean Validation MSE:             {cdl_mean_val:.6f}")
print(f"LinearGridTransformer 3-Seed Mean Validation MSE: {lgt_mean_val:.6f}")
winner = "CDLinear" if cdl_mean_val < lgt_mean_val else "LinearGridTransformer"
print(f"
>>> SELECTED OPERATIONAL FORECASTER (Validation Only): {winner} <<<")

In [ ]:
# Cell 7: Final Held-Out Evaluation on Untouched Prospective Test Set
from src.training.evaluate_tetouan import evaluate_checkpoints

for seed in [42, 123, 2026]:
    print(f"Evaluating Seed {seed} on Held-Out Test Windows...")
    evaluate_checkpoints(
        lgt_checkpoint=f"/content/drive/MyDrive/LinearGridTetouan/checkpoints/grid_colab_s{seed}_lineargridtransformer.pt",
        cdlinear_checkpoint=f"/content/drive/MyDrive/LinearGridTetouan/checkpoints/grid_colab_s{seed}_cdlinear.pt",
        artifact_dir="data/processed/grid",
        results_dir="/content/drive/MyDrive/LinearGridTetouan/results",
        output_name=f"grid_colab_s{seed}_eval.json",
    )


In [ ]:
# Cell 8: Summary Report & Statistical Significance (Diebold-Mariano Test)
import numpy as np
from scipy import stats

# Load reports and print final test comparison table
print("
" + "="*70)
print("FINAL HELD-OUT TEST METRICS (UNBIASED GENERALIZATION ESTIMATE)")
print("="*70)
for seed in [42, 123, 2026]:
    report = json.loads((results_dir / f"grid_colab_s{seed}_eval.json").read_text())
    cdl_mae = report["metrics"]["models"]["CDLinear"]["macro_average_across_zones"]["mae"]
    lgt_mae = report["metrics"]["models"]["LinearGridTransformer"]["macro_average_across_zones"]["mae"]
    base_mae = report["metrics"]["baselines"]["persistence"]["macro_average_across_zones"]["mae"]
    print(f"Seed {seed:4d} | CDLinear MAE: {cdl_mae:.2f} MW | LGT MAE: {lgt_mae:.2f} MW | Persistence: {base_mae:.2f} MW")
print("
Checkpoints and experiment logs safely persisted to Google Drive!")